In [ ]:
## Packages Import
%matplotlib widget
import copy
import time
import numpy             as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

In [ ]:
class VLMPanel:
    """
    Vortex Lattice Method panel with ring vortex
    """
    #--------------------#
    #   Initialization   #
    #--------------------#
    def __init__(self, p, v, w, b):
        """
        Panel is of quadrilateral shape and vertices must be ordered from root
        position on leading-edge proceeding in counterclockwise direction.
        The pannel can be a wing pannel (with an offset ring) or a wake pannel (with the ring being the pannel)
        """
        # Wing geometry
        self.span = b       # wing span [m]

        # nature of the pannel (wing or wake)
        self.nature = w        # 0 for wing, 1 for wake

        # Panel geometry
        self.pnt    = p        # panel vertices
        self.ctr    = None     # control point
        self.normal = None     # normal direction
        self.chord  = None     # chord length [m]
        self.width  = None     # pannel width [m]
        self.area   = None     # panel area   [m**2]
        
        # Vortex parameters
        self.vrt = v            # ring vortex points

        if self.nature == 0 :
            self._get_panel_geom()
      
    #-------------#
    #   Methods   #
    #-------------#
    def _get_panel_geom(self):
        """
        Compute panel geometric parameters such as control point position, normal
        versor, chord length, width and area, starting from panel's vertices.
        """
        p = self.pnt
        v = self.vrt
        # Compute representative panel's geometric parameters
        c_avg = 0.5 * (p[2] - p[1] + p[3] - p[0])  # average chord vector
        w_avg = 0.5 * (p[2] - p[3] + p[1] - p[0])  # average span vector
        
        # Compute position of control point
        cp = (p[0] + p[1] + 3*p[3] + 3*p[2])/8    # three-quarter line

        # Compute normal versor and panel surface
        ai = np.zeros(3)
        for i, pi in enumerate(p):
            qi = p[(i + 1) % len(p)]
            ai += np.cross(pi, qi)
        av = 0.5 * ai
        a = np.linalg.norm(av)
        if a > 0.0:
            n = av / a
        else:
            #Degenerate cells with zero area
            print(f"Degenerate panel {i} with {len(p)} vertices")


        self.ctr    = cp
        self.normal = n
        self.chord  = c_avg
        self.width  = w_avg
        self.area   = a


In [ ]:
class VLMSolver:
    """
    Vortex Lattice Method solver
    """
    #--------------------#
    #   Initialization   #
    #--------------------#
    def __init__(self, b, c, alpha, lamb, delta, phi, sym, space,
                u_inf, n, m):
        # Wing geometry
        self.span     = b       # wing span                 [m]
        self.chord    = c       # chord lengths             [m]
        self.aoa      = alpha   # angle of attack           [rad]
        self.sweep    = lamb    # quarter-chord sweep angle [rad]
        self.dihedron = delta   # dihedral angle            [rad]
        self.twist    = phi     # twist angle               [rad]
        self.symmetric= sym     # symmetric wing
        self.spacing  = space

        # Flow properties
        self.U = u_inf      # inflow velocity [m/s]

        # Discretization
        self.N = n                  # number of panels in spanwise direction
        self.M = m                  # number of panels in chordwise direction
        self.wing_panels = None     # wing panel objects
        self.wing        = None     # wing ring corner points
        self.wake_panels = None     # wake panel objects
        self.wake        = []       # wake corner points
        self.left_wake   = None     # left wake corner points
        self.right_wake  = None     # right wake corner points
        self.control     = None     # control points
        self.normal      = None     # normals

        # Linear system
        self.A = None       # influence coefficient matrix
        self.b = None       # right hand side
        self.gamma      = None   # wing vorteces strentgh
        self.gamma_wake = []   # wake vorteces strentgh

        self.time=0
    #-------------#
    #   Methods   #
    #-------------#
    def _unit_rot(self, v, theta, ax):
        """
        Elemental clockwise rotation of the reference system axes.
        
        Input:
        v     -> vector componets - shape: (N, 3)
        theta -> rotation angle - clockwise rotation: theta>0
        ax    -> rotation axis index

        Output:
        v_rot -> rotated vector components - shape: (N, 3)
        """
        # Assemble rotation matrix
        rot = np.zeros((3, 3))  # rotation matrix, v[0
        # diagonal elements
        rot[ax, ax] = 1.0
        rot[ax-1, ax-1] = np.cos(theta)
        rot[ax-2, ax-2] = np.cos(theta)
        # extra-diagonal elements
        rot[ax-2, ax-1] = -np.sin(theta)
        rot[ax-1, ax-2] = np.sin(theta)

        # Apply rotation
        v_rot = np.sum(rot[None, :, :] * v[:, None, :], axis=2)
        return v_rot

    def _symmetry(self, grid, merge, norm, p, n):
        """
        Symmetry of the geometry with respect to a plane, and merge of the two sides if needed.
        If we merge we assume that the plane pass by the origin and that the grid start from the origin

        Input:
        grid  -> grid points 
        merge -> boolean, if True the two sides are merged together, otherwise they are kept separate
        norm  -> normal vector of the symmetry plane
        p     -> point on the symmetry plane
        n     -> number of panels in spanwise direction, only needed for the merge

        Output:
        grid_sym -> symmetric grid points or the merged grid points if merge is True
        """
        S = np.eye(3) - 2*np.outer(norm, norm)/(norm @ norm)    # symmetry matrix
        grid_off = grid - p                                     # grid with the plane as origin
        grid_sym = grid_off@(S.T)                               # symmetric grid
        grid_sym = grid_sym + p                                 # symmetric grid with the original plane position
        if merge :
            m = round(np.size(grid)/(3*(n+1)))-1       # number of panels in chordwise direction
            new_grid = np.empty((0, 3))
            for i in range(m+1):
                new_grid = np.concatenate([new_grid, grid_sym[i*(n+1):(i+1)*(n+1),:][::-1][:-1], grid[i*(n+1):(i+1)*(n+1),:]])  # merge the two sides
            grid_sym = new_grid
        return grid_sym

    def _build_mesh(self):
        """
        Discretization of the wing by a finite number of lattices.
        """
        m = self.M
        n = self.N
        b     = self.span
        alpha = self.aoa
        lamb  = self.sweep
        delta = self.dihedron
        phi   = self.twist
        c     = self.chord

        # Leading-edge length of the semi-wing
        le = (0.5 * b)
        s_min, s_max = 0.0, le

        # Spanwise wing discretization, taking the symmetry and spacing parameters into account
        if self.spacing :
            if self.symmetric :                                 # if it's symmetric, we cluster only at the wing tip, otherwise we cluster at both the root and the tip
                theta = np.linspace(np.pi/2, np.pi, (n+1))
                s = le*(-np.cos(theta))                         # leading-edge coordinate 
            else :
                theta = np.linspace(0, np.pi, (n+1))
                s = le*(1-np.cos(theta))/2                      # leading-edge coordinate
        else :
            s = np.linspace(s_min, s_max, (n+1))                # leading-edge coordinate
        s = np.tile(s,m+1)                                      # grid points z components

        # Chordwise wing panels discretiration
        i = np.arange(m+1)[:, None]
        j = c[None, :]
        ch = (i/m) * j                                          # chordwise discretization at each spanwise station
        one = np.ones_like(i)
        ch = ch - one*(j/2) + c[0]/2                            # grid points x components, the chord law is centered  
        ch = ch.reshape(-1)

        # applying the twist
        phi_j = 2*s*phi/b                                                           # spanwise twist distribution
        x_quarter = c[0]/2 - c/4                                                    # usually the twist is applied around the quarter chord point
        x_quarter = (one*x_quarter[None, :]).reshape(-1)                            # broadcasting, each j section got its quarter chord position
        p_quarter = np.column_stack((ch - x_quarter, np.zeros_like(s), s))          # grid where each j section is in the local frame with the quarter chord point as x origin
        idx = np.arange((m+1)*(n+1)).reshape(m+1, n+1)
        for j in range(n+1):    
            p_quarter[idx[:,j]] = self._unit_rot(p_quarter[idx[:,j]], -phi_j[j], 2)                   # twisted frame for each section

        # Change panels corner points of frame
        p_wing  = np.column_stack((p_quarter[:,0] + x_quarter, p_quarter[:,1], p_quarter[:,2]))       # wing frame
        p_swept = np.column_stack((p_wing[:,2]*np.tan(lamb)+p_wing[:,0],p_wing[:,1],p_wing[:,2]))     # swept frame
        p_yaw   = self._unit_rot(p_swept, -delta, 0)                                                  # yawed frame
        p_earth = self._unit_rot(p_yaw,  -alpha, 2)                                                   # earth frame

        # Ring vertices
        r_wing = copy.copy(p_wing)
        for j in range(n+1):
            for i in range(m):
                r_wing[i*(n+1)+j][0] += (r_wing[(i+1)*(n+1)+j][0]-r_wing[i*(n+1)+j][0]) * 0.25        # one quarter chord offset
            r_wing[m*(n+1)+j][0] += (r_wing[m*(n+1)+j][0]-r_wing[(m-1)*(n+1)+j][0])/3
        # Change of frame
        r_swept = np.column_stack((r_wing[:,2]*np.tan(lamb)+r_wing[:,0],r_wing[:,1],r_wing[:,2]))     # swept frame
        r_yaw   = self._unit_rot(r_swept, -delta, 0)                                                  # yawed frame
        r_earth = self._unit_rot(r_yaw,  -alpha, 2)                                                   # earth frame

        # take care of the symmetry
        if self.symmetric :
            p_earth = self._symmetry(p_earth, merge=True, norm=np.array([0,0,1]), p=np.array([0,0,0]), n=n)
            r_earth = self._symmetry(r_earth, merge=True, norm=np.array([0,0,1]), p=np.array([0,0,0]), n=n)
            self.N  = 2*n
            n       = self.N           # double n for the rest of the code
        self.wing = r_earth
        panels = []     # list of panel objects
        for i in range(m):
            for j in range(n):
                pnt = [p_earth[i*(1+n)+j],
                    p_earth[i*(n+1)+j+1],
                    p_earth[(i+1)*(n+1)+j+1],
                    p_earth[(i+1)*(n+1)+j]]    # ordered panel vertices    
                vtx = [r_earth[i*(1+n)+j],
                    r_earth[i*(n+1)+j+1],
                    r_earth[(i+1)*(n+1)+j+1],
                    r_earth[(i+1)*(n+1)+j]]    # ordered ring vertices
                panels.append(VLMPanel(p=pnt, v=vtx, w=0, b=b))
        self.wing_panels = panels
        self.control     = np.array([ni.ctr for ni in self.wing_panels])
        self.normal      = np.array([ni.normal for ni in self.wing_panels])     # we compute the normals and control points vector only once

    def _induced_velocity (self,p,c1,c2,gamma):
        """
        Return the velocity induced by the vortex segments [c1, c2], of strentgh gamma,
        at the points p

        Input :
        p     -> points where the velocity is computed, shape (N,3)
        c1    -> starting points of the segments, shape (M,3)
        c2    -> ending points of the segments, shape (M,3)
        gamma -> circulation of each segment, shape (M,)

        Output :
        v     -> velocity induced at each point by each segment, shape (N,3)
        """
        t=time.time()
        r1 = p[:, None, :] - c1[None, :, :]    # (N,M,3)
        r2 = p[:, None, :] - c2[None, :, :]    # (N,M,3)
        r0 = c2[None, :, :] - c1[None, :, :]   # (1,M,3)
        r1_norm = np.linalg.norm(r1, axis=2)   # (N,M)
        r2_norm = np.linalg.norm(r2, axis=2)
        cross = np.cross(r1, r2)                # (N,M,3)
        cross_norm2 = np.sum(cross**2, axis=2)  # (N,M)
        # The safe are needed to avoid division by zero, the value of 1.0 is arbitrary since we will set the velocity to zero in these case
        eps = 1e-12
        r2_norm_safe = np.where(r2_norm < eps, 1.0, r2_norm)    
        r1_norm_safe = np.where(r1_norm < eps, 1.0, r1_norm)    
        cross_norm2_safe = np.where(cross_norm2 < eps, 1.0, cross_norm2)
        mask = ((r1_norm > eps) &(r2_norm > eps) &(cross_norm2 > eps))                               # points on or aligned with the segment
        term = np.sum(r0 * (r1 / r1_norm_safe[:, :, None] - r2 / r2_norm_safe[:, :, None]), axis=2)  # (N,M)
        coeff = gamma[None, :] / (4 * np.pi)
        v = coeff[:, :, None] * (cross / cross_norm2_safe[:, :, None]) * term[:, :, None]
        v[~mask] = 0                    # if one point is on or aligned with the segment, its induced velocity is 0
        self.time+=time.time() - t
        return np.sum(v, axis=1)  # (N,3)
    

    def _vectorize (self, grid, gamma, n) :
        """
        Transform the grid corner points into two array of points that define all
        the segments, needed for the _induced_velocity function, and the local circulation
        of each segment

        Input :
        grid  -> 2D grid of the points that defines the segments which induce the velocity
        gamma -> circulation of each vortex ring
        n     -> number of panels in spanwise direction, needed to reshape the grid and gamma in the right way

        Output :
        c1, c2    -> starting and ending points of each segment
        gamma_seg -> local circulation of each segment
        """
        m = round(np.size(grid)/(3*(n+1)))-1                    # number of panels in chordwise direction
        idx = np.arange((m+1)*(n+1)).reshape(m+1, n+1)          # reshape for building the segments points
        c1_j = grid[idx[:, :-1]]                                # spanwise segments
        c2_j = grid[idx[:, 1:]]
        c1_i = grid[idx[:-1, :]]                                # chordwise segments
        c2_i = grid[idx[1:, :]]
        c1 = np.concatenate([c1_j.reshape(-1,3), c1_i.reshape(-1,3)])
        c2 = np.concatenate([c2_j.reshape(-1,3), c2_i.reshape(-1,3)])
        gamma = gamma.reshape(m, n)
        gamma_seg_j =  gamma[1:, :] - gamma[:-1, :]                                     # local circulation at each spanwise segment except the first and last one
        gamma_seg_j_full = np.vstack([gamma[0, :], gamma_seg_j, -gamma[-1, :]])         # all the local circulations of spanwise segments
        gamma_seg_i =  -gamma[:, 1:] + gamma[:, :-1]                                    # local circulation at each chordwise segment except the first and last one        
        gamma_seg_i_full = np.hstack([-gamma[:, [0]], gamma_seg_i, gamma[:, [-1]]])     # all the local circulations of chordwise segments
        gamma_seg = np.concatenate([gamma_seg_j_full.reshape(-1), gamma_seg_i_full.reshape(-1)])    # local circulation of each segment
        return c1, c2, gamma_seg
    

    def _build_A (self):
        """
        Construct the influence coefficients matrix A by computing the velocity induced at each control point by each vortex ring,
        and projecting it on the normal direction of the panel
        """
        m, n = self.M, self.N
        ctrl = self.control
        norm = self.normal
        A = np.zeros((n*m,n*m))
        for j, vj in enumerate(self.wing_panels):
            v = vj.vrt
            c1 = np.array([v[0], v[1],v[2], v[3]]).reshape(-1,3)                
            c2 = np.array([v[1], v[2],v[3], v[0]]).reshape(-1,3)
            v_ring = self._induced_velocity(ctrl, c1, c2, np.array([1,1,1,1]))  # velocity induced at each control point by the vortex ring of panel j
            A[:,j] = np.sum(v_ring * norm, axis=1)                              # projection of the induced velocity on the normal direction of each panel
        self.A = A


    def _build_b (self, gamma_wake, gamma_l_wake, gamma_r_wake):
        """ 
        Construct the RHS b

        Input :
            -> gamma_wake the vortex strentgh of each wake ring
        """
        m, n   = self.M, self.N
        wake   = self.wake
        l_wake = self.left_wake
        r_wake = self.right_wake
        u_inf  = self.U
        ctrl   = self.control
        u_inf  = np.tile(u_inf, (m*n,1))   # m*n would return a 1D array [u_x, u_y, u_z, u_x, u_y, u_z, ...] the tuple parameters is needed to reshape it in a (m*n, 3) array
        normal = self.normal
        b = np.zeros(m*n)
        if np.size(wake) == 0 :
            v = np.tile(np.array([0,0,0]), (m*n,1))
        else :
            c1, c2, gamma_v      = self._vectorize(wake, gamma_wake, n)
            c1_l, c2_l, gamma_vl = self._vectorize(l_wake, gamma_l_wake, round(gamma_r_wake.size/m))
            c1_r, c2_r, gamma_vr = self._vectorize(r_wake, gamma_r_wake, round(gamma_r_wake.size/m))
            v = ( self._induced_velocity(ctrl, c1, c2, gamma_v)                 # velocity induced at each control point by the wake vortex rings
                 + self._induced_velocity(ctrl, c1_l, c2_l, gamma_vl)
                 + self._induced_velocity(ctrl, c1_r, c2_r, gamma_vr) )
        b = -np.sum((v + u_inf) * normal, axis=1)
        self.b = b
    
    def _kuttas_loads(self):
        """
        Compute the loads by applying the Kutta-Joukowski theorem to each vortex segment, and summing up all the contributions

        Output :
        F_tot -> total loads on the wing, shape (3,)
        """
        m, n    = self.M, self.N
        u       = self.U
        wake    = self.wake
        l_wake  = self.left_wake
        r_wake  = self.right_wake
        wing    = self.wing
        gamma   = self.gamma
        gamma_w = self.gamma_wake
        u_inf   = np.tile(u, (n*m, 1))          # m*n would return a 1D array [u_x, u_y, u_z, u_x, u_y, u_z, ...] the tuple parameters is needed to reshape it in a (m*n, 3) array
        u_inf_t = np.tile(u, ((n-1)*m, 1))      # n-1 because at both tip the local circulation is none (we assume steady state for the loads computation)
        circ    = gamma - np.concatenate([np.zeros(n), gamma[:n*(m-1)]])                            # circulation at each bound segment
        circ_t  = gamma.reshape(m, n)[:,1:].reshape(-1) - gamma.reshape(m, n)[:,:-1].reshape(-1)    # circulation at each trailing segment
        width   = []         
        points  = []         
        chords  = []         
        points_t = []        
        for k,p in enumerate(self.wing_panels):
            width.append(p.vrt[1]-p.vrt[0])             # width of each bound segment
            points.append((p.vrt[0] + p.vrt[1])/2)      # mid bound segment points
            if k%n != 0 :                                   # tips are not taken into account
                chords.append(p.vrt[0] - p.vrt[3])          # length of each trailing segment
                points_t.append((p.vrt[0] + p.vrt[3])/2)    # mid trailing segment points
        width  = np.array(width)
        points = np.array(points)
        chords = np.array(chords)
        points_t = np.array(points_t)
        n_w = round(len(l_wake)/(m+1))-1                        # number of panels in chordwise direction for the tip wake
        grid      = np.concatenate([wing[:m*(n+1)], wake])
        gamma_tot = np.concatenate([gamma, gamma_w[0]])
        c1, c2, gamma_v      = self._vectorize(grid, gamma_tot, n)        # wing + trailing edge wake
        c1_l, c2_l, gamma_vl = self._vectorize(l_wake, gamma_w[1], n_w)   # left tip wake
        c1_r, c2_r, gamma_vr = self._vectorize(r_wake, gamma_w[2], n_w)   # right tip wake
        c1 = np.concatenate([c1, c1_l, c1_r])
        c2 = np.concatenate([c2, c2_l, c2_r])
        gamma_v = np.concatenate([gamma_v, gamma_vl, gamma_vr])
        F_trailing = np.cross(u_inf_t + self._induced_velocity(points_t, c1, c2, gamma_v), circ_t[:,None]*chords)       # loads of the trailing segments by Kutta-Joukowski
        F_bound = np.cross(u_inf + self._induced_velocity(points, c1, c2, gamma_v), circ[:,None]*width)                 # loads of the bound segments by Kutta-Joukowski
        F_tot   = np.sum( np.concatenate([F_bound,F_trailing]), axis = 0 ) 
        return F_tot
        
    def _secondary_computation (self):
        """ 
        Compute the loads using the pressure difference between the two sides of each panel, and summing up all the contributions

        Output :
        F_tot -> total loads on the wing, shape (3,)
        """
        wing    = self.wing
        m, n    = self.M, self.N
        u_inf   = self.U
        wake    = self.wake
        l_wake  = self.left_wake
        r_wake  = self.right_wake
        gamma   = self.gamma
        gamma_w = self.gamma_wake
        ctrl    = self.control
        normals = self.normal
        panels  = self.wing_panels
        u_inf   = np.tile(u_inf, (n*m, 1))
        dp      = []
        d_gam_i = gamma - np.concatenate([np.zeros(n), gamma[:n*(m-1)]])                    # circulation at each bound segment      
        d_gam_j = gamma - np.hstack([np.zeros((m,1)), gamma.reshape(m, n)[:,:-1]]).reshape(-1)  # circulation at each trailing segment
        taux_i  = np.array([p.chord/(np.linalg.norm(p.chord)**2) for p in panels])          # chordwise unit vector divided by the mean chord length, for each panel
        taux_j  = np.array([p.width/(np.linalg.norm(p.width)**2) for p in panels])          # spanwise unit vector divided by the mean width, for each panel
        S       = np.array([p.area for p in panels])                                        # area of each panel
        n_w = round(len(l_wake)/(m+1))-1                            # number of panels in chordwise direction for the tip wake
        grid      = np.concatenate([wing[:m*(n+1)], wake])
        gamma_tot = np.concatenate([gamma, gamma_w[0]])
        c1, c2, gamma_v      = self._vectorize(grid, gamma_tot, n)        # trailing edge wake
        c1_l, c2_l, gamma_vl = self._vectorize(l_wake, gamma_w[1], n_w)   # left tip wake
        c1_r, c2_r, gamma_vr = self._vectorize(r_wake, gamma_w[2], n_w)   # right tip wake
        c1 = np.concatenate([c1, c1_l, c1_r])
        c2 = np.concatenate([c2, c2_l, c2_r])
        gamma_v = np.concatenate([gamma_v, gamma_vl, gamma_vr])
        V       = u_inf + self._induced_velocity(ctrl, c1, c2, gamma_v)
        dp      = np.sum(V * (taux_i*d_gam_i[:,None] + taux_j*d_gam_j[:,None]), axis=1) 
        dF      = -dp[:,None]*S[:,None]*normals
        F_tot   = np.sum(dF, axis = 0)
        return F_tot

    def _time_sim(self, t, dt, distribution):
        """ 
        Do the time stepping simulation with the wake relaxation

        Input :
        t            -> total simulation time
        dt           -> time step length
        distribution -> type of time step distribution
        """
        # Initialization
        u_inf = self.U
        m, n  = self.M, self.N
        # we memorize the wake ring strentgh
        gamma_wake   = []  
        gamma_l_wake = np.array([])
        gamma_r_wake = np.array([])  
        self._build_mesh()
        n = self.N
        wing = self.wing
        self._build_A()
        A = self.A
        inv_A = np.linalg.inv(A)
        self._build_b(gamma_wake, [], [])
        b = self.b
        Gamma = inv_A @ b                  
        Te     = wing[-(n+1):]                                    # get the trailing edge for the shedding
        L_tip  = wing.reshape(m+1, n+1, 3)[:,0,:].reshape(m+1,3)  # get the left tip
        R_tip  = wing.reshape(m+1, n+1, 3)[:,n,:].reshape(m+1,3)  # get the left tip
        wake   = np.copy(Te)
        l_wake = np.copy(L_tip)   
        r_wake = np.copy(R_tip)
        match distribution :
            case "classic" :
                dta = np.tile(dt, round(t/dt))
            case "cosine" :         # more pannels at the beginning of the simulation, to better capture the starting vortex
                theta = np.linspace(0, np.pi/2, round(t/dt))  
                dta = t*(1-np.cos(theta))
                dta = dta - np.concatenate([np.array([0]), dta[:-1]])
        for s in range(round(t/dt)):
            # simulate the wing advancement
            wake   = wake + dta[s]*u_inf
            l_wake = l_wake + dta[s]*u_inf
            r_wake = r_wake + dta[s]*u_inf
            # shed one row of ring
            wake   = np.concatenate([Te, wake])
            l_wake = np.hstack([l_wake.reshape(m+1, s+1, 3), L_tip.reshape(m+1, 1, 3)]).reshape(-1, 3)
            r_wake = np.hstack([R_tip.reshape(m+1, 1, 3), r_wake.reshape(m+1, s+1, 3)]).reshape(-1, 3)
            # store the vorteces strentgh of the wake
            gamma_wake   = np.concatenate([Gamma[-(n):], gamma_wake])
            gamma_l_wake = np.hstack([gamma_l_wake.reshape(m, s), Gamma.reshape(m, n)[:,0].reshape(m, 1)]).reshape(-1)
            gamma_r_wake = np.hstack([Gamma.reshape(m, n)[:,n-1].reshape(m, 1), gamma_r_wake.reshape(m, s)]).reshape(-1)
            self.wake       = wake
            self.left_wake  = l_wake
            self.right_wake = r_wake
            # update the right hand side and gamma copmutation
            self._build_b(gamma_wake, gamma_l_wake, gamma_r_wake)
            Gamma = inv_A @ self.b 
            # simulate the wake rollup
            grid      = np.concatenate([wing[:m*(n+1)], wake])
            gamma_tot = np.concatenate([Gamma, gamma_wake])
            c1, c2, gamma_v      = self._vectorize(grid, gamma_tot, n)          # wing + trailing edge wake
            c1_l, c2_l, gamma_vl = self._vectorize(l_wake, gamma_l_wake, s+1)   # left tip wake
            c1_r, c2_r, gamma_vr = self._vectorize(r_wake, gamma_r_wake, s+1)   # right tip wake
            c1 = np.concatenate([c1, c1_l, c1_r])
            c2 = np.concatenate([c2, c2_l, c2_r])
            gamma_v = np.concatenate([gamma_v, gamma_vl, gamma_vr])
            wake   = wake + np.concatenate([np.zeros_like(Te), dta[s]*self._induced_velocity(wake[n+1:], c1, c2, gamma_v)])    # each corner point except at the edges are convect by the induced velocity
            l_wake = l_wake + np.hstack([
                dta[s]*self._induced_velocity(l_wake.reshape(m+1, s+2, 3)[:,:s+1].reshape(-1, 3), c1, c2, gamma_v).reshape(m+1, s+1, 3),
                np.zeros_like(L_tip.reshape(m+1,1,3))
                ]).reshape(-1, 3)                               # reshape in 2D grid to not take into accountthe tip edge
            r_wake = r_wake + np.hstack([
                np.zeros_like(R_tip.reshape(m+1,1,3)), 
                dta[s]*self._induced_velocity(r_wake.reshape(m+1, s+2, 3)[:,1:].reshape(-1, 3), c1, c2, gamma_v).reshape(m+1, s+1, 3),
                ]).reshape(-1, 3)
        self.wake       = wake
        self.left_wake  = l_wake
        self.right_wake = r_wake
        self.gamma      = Gamma
        self.gamma_wake = [gamma_wake, gamma_l_wake, gamma_r_wake]
        # build wake_panels, usefull for plotting
        panels = []
        for i in range(round(len(wake)/(n+1))-1):
            for j in range(n):
                pnt = [wake[i*(1+n)+j],
                    wake[i*(n+1)+j+1],
                    wake[(i+1)*(n+1)+j+1],
                    wake[(i+1)*(n+1)+j]]    # trailing edge wake panel vertices 
                panels.append(VLMPanel(p=pnt, v=pnt, w=1, b=b))
        n = round(len(l_wake)/(m+1))-1
        for i in range(m):
            for j in range(n):
                pnt = [l_wake[i*(1+n)+j],
                    l_wake[i*(n+1)+j+1],
                    l_wake[(i+1)*(n+1)+j+1],
                    l_wake[(i+1)*(n+1)+j]]    # left wake panel vertices
                panels.append(VLMPanel(p=pnt, v=pnt, w=1, b=b))
                pnt = [r_wake[i*(1+n)+j],
                    r_wake[i*(n+1)+j+1],
                    r_wake[(i+1)*(n+1)+j+1],
                    r_wake[(i+1)*(n+1)+j]]    # right wake panel vertices  
                panels.append(VLMPanel(p=pnt, v=pnt, w=1, b=b))
        self.wake_panels = panels



In [ ]:
## Utilities
def chord_fn(n, ar, b, sym, space, shape, lam=2):
    """
    Return chord length distribution along spanwise direction according to the
    desired planform shape.

    Input:
        n     -> number of spanwise stations
        ar    -> aspect ratio
        b     -> wing span
        sym   -> symmetric or not
        space -> uniform or cosine spanwise spacing
        shape -> wing planform shape, can be "rectangular", "elliptical" or "tapered"
        lam   -> taper ratio, only needed for tapered shape

    Output:
        c     -> chord length distribution along spanwise direction, shape (n,)
    """
    le = (0.5 * b)
    s_min, s_max = 0.0, le
    if space :
        if sym :
            theta = np.linspace(np.pi/2, np.pi, n)
            s = le*(-np.cos(theta))
        else :
            theta = np.linspace(0, np.pi, n)
            s = le*(1-np.cos(theta))/2
    else :
        s = np.linspace(s_min, s_max, n)
    match shape:
        case "rectangular":
            c = b/ar * np.ones(n)
        case "elliptical":
            c = 4*b/(np.pi*ar) * np.sqrt(1 - (s/le)**2)
        case "tapered":
            c = - s*4*(lam - 1)/(ar*(lam + 1)) + 4*le*lam/(ar*(lam + 1))
    return c




In [ ]:
## Parameters Definition

# Wing geometry
B = 1        # wing span                 [m]
AR = 5       # aspect ration             [-]
ALPHA  = 10  # angle of attack           [deg] - positive definite for counterclockwise rotations about z-axis
LAMBDA = 0   # middle-chord sweep angle  [deg] - positive definite for counterclockwise rotations about x-axis  Y-AXIS
DELTA  = 0    # dihedral angle            [deg] - positive definite for rotations oriented towards positive y-axis  X-AXIS, négatif pour dyhedre classique 
PHI    = 0    # twist tip angle           [deg] - negative for washout

SYM   = True           # symmetric wing configuration
SPACE = True          # spacing distribution, False = uniform, True = cos
SHAPE = "elliptical"   # shape of the wing
TIME  = "classic"       # time step distribution - classic or cosine at the starting vortex

# Flow properties
U = 1.0     # inflow velocity [m/s]

# Wing discretization
N = 15      # number of panels in spanwise direction
M = 4      # number of panels in chordwise direction

# Time simulation parameters
T  = 4   # length of the simulation in s
DT = 0.1    # time step lentgh 




In [ ]:
## Time stepping algorithm 
vlm = VLMSolver(b=B,
                c=chord_fn((N+1), AR, B, SYM, SPACE, SHAPE),
                alpha=np.deg2rad(ALPHA),
                lamb=np.deg2rad(LAMBDA),
                delta=np.deg2rad(DELTA),
                phi=np.deg2rad(PHI),
                sym=SYM,
                space=SPACE,
                u_inf=np.array([U, 0.0, 0.0]),
                n=N,m=M)
#vlm._build_mesh()
t0 = time.time()
vlm._time_sim(t=T, dt=DT, distribution=TIME)
t1 = time.time()
#plot
fig_3d = plt.figure(figsize=(14, 8),constrained_layout=True)
fig    = plt.figure(figsize=(10,4))
ax1 = fig.add_subplot()
ax4 = fig_3d.add_subplot(111, projection='3d')
for i, panel in enumerate(vlm.wing_panels):
        pnt    = copy.copy(panel.pnt)
        vrt    = copy.copy(panel.vrt)
        # Close the polygon shape
        pnt.append(pnt[0])
        pnt_plt = np.array(pnt)
        vrt.append(vrt[0])
        vrt_plt = np.array(vrt)

        # Plot lattice
        if i==0 :               # In order to have just one legend
                ax1.plot(pnt_plt[:, 2], pnt_plt[:, 0], color='black', label='Wing panels')
                ax1.plot(vrt_plt[:, 2], vrt_plt[:, 0], color='red', lw = 0.8, label='Vortex rings')
                # 3D plot
                ax4.plot(pnt_plt[:, 2], pnt_plt[:, 0], pnt_plt[:, 1], color='black', label='Wing panels')
                ax4.plot(vrt_plt[:, 2], vrt_plt[:, 0], vrt_plt[:, 1], color='red', lw=0.8, label='Vortex rings')
        else :
                ax1.plot(pnt_plt[:, 2], pnt_plt[:, 0], color='black')
                ax1.plot(vrt_plt[:, 2], vrt_plt[:, 0], color='red', lw = 0.8)
                # 3D plot
                ax4.plot(pnt_plt[:, 2], pnt_plt[:, 0], pnt_plt[:, 1], color='black')
                ax4.plot(vrt_plt[:, 2], vrt_plt[:, 0], vrt_plt[:, 1], color='red', lw=0.8)
for i, panel in enumerate(vlm.wake_panels):
        pnt    = copy.copy(panel.pnt)
        # Close the polygon shape
        pnt.append(pnt[0])
        pnt_plt = np.array(pnt)
        # Plot lattice
        if i==0 :               # In order to have just one legend
                # 3D plot
                ax4.plot(pnt_plt[:, 2], pnt_plt[:, 0], pnt_plt[:, 1], color='blue', lw=0.6, label='Wake panels')
        else :
                # 3D plot
                ax4.plot(pnt_plt[:, 2], pnt_plt[:, 0], pnt_plt[:, 1], color='blue', lw=0.6)    

x_lim = ax1.get_xlim()
y_lim = ax1.get_ylim()
ax1.set_xlabel("$z$")
ax1.set_ylabel("$x$")
ax1.set_xlim(x_lim[0],x_lim[1])
ax1.set_ylim(y_lim[1],y_lim[0])
ax1.axis('equal')
ax1.legend()
ax4.view_init(elev=30, azim=40)
x_lim = ax4.get_xlim3d()
y_lim = ax4.get_ylim3d()
z_lim = ax4.get_zlim3d()
ax4.set_xlim(x_lim[0],x_lim[1])
ax4.set_ylim(y_lim[0],y_lim[1]) 
ax4.set_zlim(z_lim[0],z_lim[1])
ax4.set_box_aspect([-x_lim[0]+x_lim[1],-y_lim[0]+y_lim[1],-z_lim[0]+z_lim[1]])
ax4.set_xlabel("z", fontstyle='italic')
ax4.set_ylabel("x", fontstyle='italic')
ax4.set_zlabel("y", fontstyle='italic')
ax4.xaxis.set_major_locator(MultipleLocator(1))
ax4.zaxis.set_major_locator(MultipleLocator(0.1))

ax4.set_title('3D view')
ax4.legend()

t2 = time.time()
print("Computation time : ",t1-t0)
print("Plotting time    : ",t2-t1)
print("veloctity time   : ", vlm.time)
s = 0

l = vlm._kuttas_loads()[1]
print("Lift coefficient : ", 2*l*AR/(B**2))
print("New lift coef    : ", 2*vlm._secondary_computation()[1]*AR/(B**2))
print("Drag by F_tot    : ", 2*vlm._secondary_computation()[0]*AR/(B**2))


In [ ]:
## Cl versus alpha
alpha = np.linspace(0,11,12)
timee = np.linspace(1,8,8)
cl    = []
cl_l  = []
for ti in timee :
    vlm = VLMSolver(b=B,
                c=chord_fn((2+1), AR, B, SYM, SPACE, SHAPE),
                alpha=np.deg2rad(ALPHA),
                lamb=np.deg2rad(LAMBDA),
                delta=np.deg2rad(DELTA),
                phi=np.deg2rad(PHI),
                sym=SYM,
                space=SPACE,
                u_inf=np.array([U, 0.0, 0.0]),
                n=2,m=2)
    vlm._time_sim(ti, DT, distribution=TIME)
    l = vlm._secondary_computation()[1]
    cl.append(2*l*AR/(B**2))
    l = vlm._kuttas_loads()[1]
    cl_l.append(2*l*AR/(B**2))
print(timee)
print(cl)
print(cl_l)

